# 00 Project Setup And Shared Functions

All reusable implementation code for the notebook-only project lives here. Later notebooks use `%run 00_project_setup_and_shared_functions.ipynb` instead of importing local Python modules.


In [ ]:
# Shared imports and deterministic project paths.
import json
import math
import os
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yaml
except Exception as exc:
    yaml = None
    warnings.warn(f"PyYAML is unavailable: {exc}")

from scipy.stats import norm
from scipy.interpolate import griddata
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})


def infer_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    if (cwd / "notebooks" / "00_project_setup_and_shared_functions.ipynb").exists():
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "quantum_indian_option_risk_engine_notebooks" / "notebooks").exists():
            return parent / "quantum_indian_option_risk_engine_notebooks"
    return cwd

PROJECT_ROOT = infer_project_root()
CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SYNTHETIC_DATA_DIR = DATA_DIR / "synthetic"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
REPORTS_DIR = RESULTS_DIR / "reports"
NOTEBOOK_OUTPUTS_DIR = RESULTS_DIR / "notebook_outputs"
PAPER_DIR = PROJECT_ROOT / "paper"

for directory in [CONFIG_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, SYNTHETIC_DATA_DIR, FIGURES_DIR, TABLES_DIR, REPORTS_DIR, NOTEBOOK_OUTPUTS_DIR, PAPER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int = 20260526) -> np.random.Generator:
    np.random.seed(seed)
    return np.random.default_rng(seed)

rng = set_global_seed()
print(f"Shared notebook loaded. Project root: {PROJECT_ROOT}")


## Config And Data Utilities


In [ ]:
# Config, persistence, and option-chain data utilities.

def load_config(path: Optional[Union[str, Path]] = None) -> Dict[str, Any]:
    if yaml is None:
        raise RuntimeError("PyYAML is required for config loading.")
    config_path = Path(path) if path else CONFIG_DIR / "default_config.yaml"
    with config_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_json(obj: Any, path: Union[str, Path]) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    return output_path


def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=True)
    return path


def save_output(obj: Any, filename: str) -> Path:
    path = NOTEBOOK_OUTPUTS_DIR / filename
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    elif isinstance(obj, pd.Series):
        obj.to_csv(path)
    else:
        save_json(obj, path)
    return path


def save_current_figure(filename: str, dpi: int = 150) -> Path:
    path = FIGURES_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path


def normalize_option_type(value: Any) -> str:
    text = str(value).strip().lower()
    if text in {"ce", "c", "call", "calls"}:
        return "call"
    if text in {"pe", "p", "put", "puts"}:
        return "put"
    return text


def normalize_option_chain_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(c).strip().lower().replace(" ", "_").replace("-", "_") for c in result.columns]
    synonyms = {
        "symbol": ["symbol", "underlying", "underlying_symbol", "name"],
        "expiry": ["expiry", "expiry_date", "expiration", "expiration_date"],
        "strike": ["strike", "strike_price", "strikeprice"],
        "option_type": ["option_type", "type", "instrument_type", "right", "cp"],
        "underlying_price": ["underlying_price", "spot", "spot_price", "underlying_value", "underlyingvalue"],
        "last_price": ["last_price", "ltp", "last", "close", "market_price"],
        "bid": ["bid", "bid_price", "best_bid"],
        "ask": ["ask", "ask_price", "best_ask", "offer"],
        "implied_volatility": ["implied_volatility", "iv", "implied_vol", "volatility"],
        "open_interest": ["open_interest", "oi"],
        "volume": ["volume", "traded_volume"],
        "timestamp": ["timestamp", "date", "datetime", "valuation_date"],
    }
    rename = {}
    for canonical, choices in synonyms.items():
        for choice in choices:
            if choice in result.columns:
                rename[choice] = canonical
                break
    result = result.rename(columns=rename)
    for col in synonyms:
        if col not in result.columns:
            result[col] = np.nan
    result["option_type"] = result["option_type"].map(normalize_option_type)
    for col in ["strike", "underlying_price", "last_price", "bid", "ask", "implied_volatility", "open_interest", "volume"]:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result["expiry"] = pd.to_datetime(result["expiry"], errors="coerce")
    result["timestamp"] = pd.to_datetime(result["timestamp"], errors="coerce")
    if result["timestamp"].notna().any():
        valuation_dates = result["timestamp"].fillna(result["timestamp"].dropna().iloc[0])
    else:
        valuation_dates = pd.Timestamp("2026-05-26")
    result["maturity"] = (result["expiry"] - valuation_dates).dt.days / 365.0
    result.loc[result["maturity"].isna() | (result["maturity"] <= 0), "maturity"] = np.nan
    result["mid_price"] = np.where(result["bid"].notna() & result["ask"].notna(), (result["bid"] + result["ask"]) / 2.0, result["last_price"])
    result["source"] = result.get("source", "csv_input")
    ordered = ["symbol", "expiry", "strike", "option_type", "underlying_price", "last_price", "bid", "ask", "mid_price", "implied_volatility", "open_interest", "volume", "timestamp", "maturity", "source"]
    return result[ordered + [c for c in result.columns if c not in ordered]]


def load_option_chain_csv(path: Union[str, Path]) -> pd.DataFrame:
    return normalize_option_chain_columns(pd.read_csv(path))


def round_to_step(values: np.ndarray, step: float) -> np.ndarray:
    return np.round(values / step) * step


def generate_synthetic_option_chain(
    underlying: str,
    spot: float,
    strike_step: float,
    base_volatility: float,
    risk_free_rate: float,
    maturities_days: Sequence[int],
    strikes_each_side: int = 8,
    smile_strength: float = 0.55,
    skew_strength: float = -0.10,
    min_volatility: float = 0.08,
    max_volatility: float = 0.35,
    seed: int = 20260526,
) -> pd.DataFrame:
    local_rng = np.random.default_rng(seed + abs(hash(underlying)) % 10000)
    valuation_date = pd.Timestamp("2026-05-26 15:30:00")
    rows = []
    for days in maturities_days:
        T = days / 365.0
        moneyness_grid = np.linspace(0.82, 1.18, 2 * strikes_each_side + 1)
        strikes = np.unique(round_to_step(spot * moneyness_grid, strike_step))
        for strike in strikes:
            log_m = math.log(strike / spot)
            term_bump = 0.015 * math.sqrt(T * 12.0)
            smile = smile_strength * log_m**2
            skew = skew_strength * log_m
            iv = float(np.clip(base_volatility + term_bump + smile + skew, min_volatility, max_volatility))
            for option_type in ["call", "put"]:
                theoretical = float(black_scholes_price(spot, strike, T, risk_free_rate, iv, option_type))
                spread_pct = 0.008 + 0.030 * min(1.0, abs(log_m) * 5.0)
                spread = max(strike_step * 0.01, theoretical * spread_pct)
                bid = max(0.05, theoretical - spread / 2.0)
                ask = theoretical + spread / 2.0
                last = max(0.05, theoretical + local_rng.normal(0.0, spread * 0.15))
                rows.append({
                    "symbol": underlying,
                    "expiry": (valuation_date + pd.Timedelta(days=days)).date().isoformat(),
                    "strike": float(strike),
                    "option_type": option_type,
                    "underlying_price": float(spot),
                    "last_price": last,
                    "bid": bid,
                    "ask": ask,
                    "implied_volatility": iv,
                    "open_interest": int(local_rng.integers(500, 120000)),
                    "volume": int(local_rng.integers(10, 25000)),
                    "timestamp": valuation_date.isoformat(),
                    "source": "synthetic_indian_market_not_live",
                })
    return normalize_option_chain_columns(pd.DataFrame(rows))


def generate_synthetic_indian_market(config: Dict[str, Any]) -> pd.DataFrame:
    market = config["market"]
    synth = config["synthetic_market"]
    frames = []
    for underlying in ["NIFTY", "BANKNIFTY"]:
        uconf = market["underlyings"][underlying]
        frames.append(generate_synthetic_option_chain(
            underlying=underlying,
            spot=float(uconf["spot"]),
            strike_step=float(uconf["strike_step"]),
            base_volatility=float(uconf["base_volatility"]),
            risk_free_rate=float(market["risk_free_rate_default"]),
            maturities_days=market["maturities_days"],
            strikes_each_side=int(synth["strikes_each_side"]),
            smile_strength=float(synth["smile_strength"]),
            skew_strength=float(synth["skew_strength"]),
            min_volatility=float(synth["min_volatility"]),
            max_volatility=float(synth["max_volatility"]),
            seed=int(config["random_seed"]),
        ))
    return pd.concat(frames, ignore_index=True)


## Classical Pricing And Portfolio Utilities


In [ ]:
# Black-Scholes prices, Greeks, finite-difference Greeks, and portfolio utilities.

def _maybe_scalar(value: np.ndarray) -> Union[float, np.ndarray]:
    arr = np.asarray(value)
    return float(arr) if arr.ndim == 0 else arr


def black_scholes_price(S: Any, K: Any, T: Any, r: Any, sigma: Any, option_type: str = "call") -> Union[float, np.ndarray]:
    S, K, T, r, sigma = np.broadcast_arrays(np.asarray(S, dtype=float), np.asarray(K, dtype=float), np.asarray(T, dtype=float), np.asarray(r, dtype=float), np.asarray(sigma, dtype=float))
    eps = 1e-12
    sqrtT = np.sqrt(np.maximum(T, eps))
    sigma_safe = np.maximum(sigma, eps)
    d1 = (np.log(np.maximum(S, eps) / np.maximum(K, eps)) + (r + 0.5 * sigma_safe**2) * np.maximum(T, eps)) / (sigma_safe * sqrtT)
    d2 = d1 - sigma_safe * sqrtT
    call = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    intrinsic_call = np.maximum(S - K, 0.0)
    intrinsic_put = np.maximum(K - S, 0.0)
    call = np.where(T <= eps, intrinsic_call, call)
    put = np.where(T <= eps, intrinsic_put, put)
    price = call if normalize_option_type(option_type) == "call" else put
    return _maybe_scalar(price)


def black_scholes_greeks(S: float, K: float, T: float, r: float, sigma: float, option_type: str = "call") -> Dict[str, float]:
    eps = 1e-12
    S = float(S); K = float(K); T = max(float(T), eps); r = float(r); sigma = max(float(sigma), eps)
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    pdf_d1 = norm.pdf(d1)
    gamma = pdf_d1 / (S * sigma * sqrtT)
    vega = S * pdf_d1 * sqrtT
    common_theta = -(S * pdf_d1 * sigma) / (2.0 * sqrtT)
    if normalize_option_type(option_type) == "call":
        delta = norm.cdf(d1)
        theta = common_theta - r * K * math.exp(-r * T) * norm.cdf(d2)
        rho = K * T * math.exp(-r * T) * norm.cdf(d2)
    else:
        delta = norm.cdf(d1) - 1.0
        theta = common_theta + r * K * math.exp(-r * T) * norm.cdf(-d2)
        rho = -K * T * math.exp(-r * T) * norm.cdf(-d2)
    return {"Delta": float(delta), "Gamma": float(gamma), "Vega": float(vega), "Theta": float(theta), "Rho": float(rho)}


def finite_difference_greeks(S: float, K: float, T: float, r: float, sigma: float, option_type: str = "call") -> Dict[str, float]:
    hS = max(1e-3, abs(S) * 1e-4)
    hsig = 1e-4
    hT = min(max(1 / 3650, T * 0.01), max(T * 0.49, 1e-5))
    hr = 1e-5
    p0 = black_scholes_price(S, K, T, r, sigma, option_type)
    p_up = black_scholes_price(S + hS, K, T, r, sigma, option_type)
    p_dn = black_scholes_price(max(S - hS, 1e-8), K, T, r, sigma, option_type)
    delta = (p_up - p_dn) / (2 * hS)
    gamma = (p_up - 2 * p0 + p_dn) / (hS**2)
    vega = (black_scholes_price(S, K, T, r, sigma + hsig, option_type) - black_scholes_price(S, K, T, r, max(sigma - hsig, 1e-8), option_type)) / (2 * hsig)
    theta = (black_scholes_price(S, K, max(T - hT, 1e-8), r, sigma, option_type) - black_scholes_price(S, K, T + hT, r, sigma, option_type)) / (2 * hT)
    rho = (black_scholes_price(S, K, T, r + hr, sigma, option_type) - black_scholes_price(S, K, T, r - hr, sigma, option_type)) / (2 * hr)
    return {"Delta": float(delta), "Gamma": float(gamma), "Vega": float(vega), "Theta": float(theta), "Rho": float(rho)}


def put_call_parity_error(S: float, K: float, T: float, r: float, sigma: float) -> float:
    call = black_scholes_price(S, K, T, r, sigma, "call")
    put = black_scholes_price(S, K, T, r, sigma, "put")
    return float(call - put - (S - K * math.exp(-r * T)))


@dataclass
class OptionPosition:
    underlying: str
    option_type: str
    strike: float
    expiry: str
    quantity: float
    market_price: float
    implied_volatility: float
    risk_free_rate: float
    spot: float
    maturity: float
    label: str = ""
    source: str = "synthetic"


def coerce_position(position: Union[OptionPosition, Dict[str, Any], pd.Series]) -> OptionPosition:
    if isinstance(position, OptionPosition):
        return position
    data = dict(position)
    return OptionPosition(
        underlying=str(data.get("underlying", data.get("symbol", "UNKNOWN"))),
        option_type=normalize_option_type(data.get("option_type", "call")),
        strike=float(data.get("strike")),
        expiry=str(data.get("expiry", "")),
        quantity=float(data.get("quantity", 1.0)),
        market_price=float(data.get("market_price", data.get("mid_price", data.get("last_price", np.nan)))),
        implied_volatility=float(data.get("implied_volatility", data.get("sigma", 0.20))),
        risk_free_rate=float(data.get("risk_free_rate", data.get("r", 0.065))),
        spot=float(data.get("spot", data.get("underlying_price", data.get("S0", np.nan)))),
        maturity=float(data.get("maturity", data.get("T", np.nan))),
        label=str(data.get("label", data.get("symbol", "position"))),
        source=str(data.get("source", "unknown")),
    )


def load_portfolio_config(path: Optional[Union[str, Path]] = None) -> List[OptionPosition]:
    if yaml is None:
        raise RuntimeError("PyYAML is required for portfolio config loading.")
    portfolio_path = Path(path) if path else CONFIG_DIR / "sample_portfolio.yaml"
    with portfolio_path.open("r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    return [coerce_position(item) for item in raw["positions"]]


def positions_to_dataframe(positions: Sequence[Union[OptionPosition, Dict[str, Any]]]) -> pd.DataFrame:
    return pd.DataFrame([asdict(coerce_position(p)) for p in positions])


def value_position(position: Union[OptionPosition, Dict[str, Any]], pricing_fn: Callable[..., float] = black_scholes_price) -> Dict[str, float]:
    pos = coerce_position(position)
    theoretical = float(pricing_fn(pos.spot, pos.strike, pos.maturity, pos.risk_free_rate, pos.implied_volatility, pos.option_type))
    market_price = theoretical if not np.isfinite(pos.market_price) else pos.market_price
    greeks = black_scholes_greeks(pos.spot, pos.strike, pos.maturity, pos.risk_free_rate, pos.implied_volatility, pos.option_type)
    row = asdict(pos)
    row.update({
        "theoretical_price": theoretical,
        "position_theoretical_value": theoretical * pos.quantity,
        "position_market_value": market_price * pos.quantity,
        "position_pnl": (theoretical - market_price) * pos.quantity,
    })
    for key, value in greeks.items():
        row[key] = value
        row[f"position_{key}"] = value * pos.quantity
    return row


def value_portfolio(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], pricing_fn: Callable[..., float] = black_scholes_price) -> Tuple[pd.DataFrame, Dict[str, float]]:
    df = pd.DataFrame([value_position(p, pricing_fn=pricing_fn) for p in positions])
    totals = {
        "total_theoretical_value": float(df["position_theoretical_value"].sum()),
        "total_market_value": float(df["position_market_value"].sum()),
        "total_pnl": float(df["position_pnl"].sum()),
    }
    for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
        totals[f"portfolio_{greek}"] = float(df[f"position_{greek}"].sum())
    return df, totals


def portfolio_from_option_chain(chain: pd.DataFrame, max_positions: int = 8) -> List[OptionPosition]:
    cleaned = normalize_option_chain_columns(chain)
    selected = cleaned.dropna(subset=["strike", "underlying_price", "implied_volatility", "maturity", "mid_price"]).copy()
    selected["atm_distance"] = (selected["strike"] / selected["underlying_price"] - 1.0).abs()
    selected = selected.sort_values(["symbol", "maturity", "atm_distance"]).head(max_positions)
    quantities = np.array([75, -50, 40, -25, 100, -60, 30, -20], dtype=float)
    positions = []
    for idx, (_, row) in enumerate(selected.iterrows()):
        positions.append(coerce_position({
            "underlying": row["symbol"], "option_type": row["option_type"], "strike": row["strike"],
            "expiry": str(row["expiry"].date() if pd.notna(row["expiry"]) else ""),
            "quantity": float(quantities[idx % len(quantities)]), "market_price": row["mid_price"],
            "implied_volatility": row["implied_volatility"], "risk_free_rate": 0.065,
            "spot": row["underlying_price"], "maturity": row["maturity"],
            "label": f"{row['symbol']}_{row['option_type']}_{int(row['strike'])}", "source": "synthetic_chain_selection",
        }))
    return positions


## Risk And Finite-Difference Utilities


In [ ]:
# Stress testing, VaR/Expected Shortfall, and finite-difference PDE utilities.

def shock_position(position: Union[OptionPosition, Dict[str, Any]], spot_rel: float = 0.0, vol_rel: float = 0.0, vol_abs: float = 0.0, rate_abs: float = 0.0, days_forward: float = 0.0, spot_override: Optional[float] = None) -> OptionPosition:
    pos = coerce_position(position)
    new_spot = float(spot_override) if spot_override is not None else pos.spot * (1.0 + spot_rel)
    new_sigma = max(1e-6, pos.implied_volatility * (1.0 + vol_rel) + vol_abs)
    new_T = max(1e-6, pos.maturity - days_forward / 365.0)
    return OptionPosition(pos.underlying, pos.option_type, pos.strike, pos.expiry, pos.quantity, pos.market_price, new_sigma, pos.risk_free_rate + rate_abs, new_spot, new_T, pos.label, pos.source)


def revalue_portfolio_with_shock(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], **shock_kwargs: Any) -> Dict[str, float]:
    shocked = [shock_position(p, **shock_kwargs) for p in positions]
    _, totals = value_portfolio(shocked)
    return totals


def spot_vol_stress_matrix(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], spot_shocks: Sequence[float], vol_shocks: Sequence[float]) -> pd.DataFrame:
    _, base = value_portfolio(positions)
    base_value = base["total_theoretical_value"]
    matrix = pd.DataFrame(index=[f"{s:+.1%}" for s in spot_shocks], columns=[f"{v:+.1%}" for v in vol_shocks], dtype=float)
    for s in spot_shocks:
        for v in vol_shocks:
            shocked = revalue_portfolio_with_shock(positions, spot_rel=s, vol_rel=v)
            matrix.loc[f"{s:+.1%}", f"{v:+.1%}"] = shocked["total_theoretical_value"] - base_value
    return matrix


def standard_stress_table(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], config: Dict[str, Any]) -> pd.DataFrame:
    _, base = value_portfolio(positions)
    scenarios = []
    for spot in config["stress"]["spot_shocks"]:
        totals = revalue_portfolio_with_shock(positions, spot_rel=float(spot))
        scenarios.append({"scenario": f"spot_{spot:+.1%}", "type": "spot", "shock": spot, "pnl": totals["total_theoretical_value"] - base["total_theoretical_value"], **totals})
    for vol in config["stress"]["volatility_relative_shocks"]:
        totals = revalue_portfolio_with_shock(positions, vol_rel=float(vol))
        scenarios.append({"scenario": f"vol_rel_{vol:+.1%}", "type": "vol_relative", "shock": vol, "pnl": totals["total_theoretical_value"] - base["total_theoretical_value"], **totals})
    for vol_abs in config["stress"]["volatility_absolute_shocks"]:
        totals = revalue_portfolio_with_shock(positions, vol_abs=float(vol_abs))
        scenarios.append({"scenario": f"vol_abs_{vol_abs:+.1%}", "type": "vol_absolute", "shock": vol_abs, "pnl": totals["total_theoretical_value"] - base["total_theoretical_value"], **totals})
    for days in config["stress"]["time_decay_days"]:
        totals = revalue_portfolio_with_shock(positions, days_forward=float(days))
        scenarios.append({"scenario": f"time_decay_{days}d", "type": "time_decay", "shock": days, "pnl": totals["total_theoretical_value"] - base["total_theoretical_value"], **totals})
    for rate in config["stress"]["rate_shocks"]:
        totals = revalue_portfolio_with_shock(positions, rate_abs=float(rate))
        scenarios.append({"scenario": f"rate_{rate:+.0%}", "type": "rate", "shock": rate, "pnl": totals["total_theoretical_value"] - base["total_theoretical_value"], **totals})
    return pd.DataFrame(scenarios).sort_values("pnl")


def monte_carlo_portfolio_pnl(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], n_scenarios: int = 5000, horizon_days: float = 1.0, seed: int = 20260526) -> pd.Series:
    local_rng = np.random.default_rng(seed)
    positions = [coerce_position(p) for p in positions]
    _, base = value_portfolio(positions)
    base_value = base["total_theoretical_value"]
    underlyings = sorted({p.underlying for p in positions})
    horizon_T = horizon_days / 365.0
    shocks: Dict[str, np.ndarray] = {}
    for underlying in underlyings:
        group = [p for p in positions if p.underlying == underlying]
        spot0 = group[0].spot
        sigma = float(np.mean([p.implied_volatility for p in group]))
        r = float(np.mean([p.risk_free_rate for p in group]))
        z = local_rng.standard_normal(n_scenarios)
        shocks[underlying] = spot0 * np.exp((r - 0.5 * sigma**2) * horizon_T + sigma * math.sqrt(horizon_T) * z)
    pnl = np.zeros(n_scenarios)
    for i in range(n_scenarios):
        shocked_positions = [shock_position(p, days_forward=horizon_days, spot_override=float(shocks[p.underlying][i])) for p in positions]
        _, totals = value_portfolio(shocked_positions)
        pnl[i] = totals["total_theoretical_value"] - base_value
    return pd.Series(pnl, name="portfolio_pnl")


def var_expected_shortfall(pnl: Union[pd.Series, np.ndarray], levels: Sequence[float] = (0.95, 0.99)) -> Dict[str, float]:
    pnl_arr = np.asarray(pnl, dtype=float)
    losses = -pnl_arr
    metrics: Dict[str, float] = {}
    for level in levels:
        var = float(np.quantile(losses, level))
        tail = losses[losses >= var]
        es = float(tail.mean()) if len(tail) else var
        pct = int(round(level * 100))
        metrics[f"VaR_{pct}"] = var
        metrics[f"ES_{pct}"] = es
    return metrics


def finite_difference_black_scholes_solver(
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: str = "call",
    S_max: Optional[float] = None,
    stock_steps: int = 160,
    time_steps: int = 160,
    method: str = "crank_nicolson",
) -> Dict[str, Any]:
    option_type = normalize_option_type(option_type)
    S_max = float(S_max or 4 * K)
    M = int(stock_steps)
    N = int(time_steps)
    dS = S_max / M
    dt = T / N
    S_grid = np.linspace(0, S_max, M + 1)
    if option_type == "call":
        V = np.maximum(S_grid - K, 0.0)
    else:
        V = np.maximum(K - S_grid, 0.0)
    j = np.arange(1, M)
    if method.lower().replace("-", "_") in {"implicit", "implicit_euler"}:
        a = 0.5 * dt * (sigma**2 * j**2 - r * j)
        b = 1.0 + dt * (sigma**2 * j**2 + r)
        c = 0.5 * dt * (sigma**2 * j**2 + r * j)
        A = diags([-a[1:], b, -c[:-1]], offsets=[-1, 0, 1], format="csc")
        for step in range(N):
            tau_new = (step + 1) * dt
            low_new = K * math.exp(-r * tau_new) if option_type == "put" else 0.0
            high_new = 0.0 if option_type == "put" else S_max - K * math.exp(-r * tau_new)
            rhs = V[1:M].copy()
            rhs[0] += a[0] * low_new
            rhs[-1] += c[-1] * high_new
            V[1:M] = spsolve(A, rhs)
            V[0] = low_new; V[M] = high_new
    else:
        alpha = 0.25 * dt * (sigma**2 * j**2 - r * j)
        beta = -0.5 * dt * (sigma**2 * j**2 + r)
        gamma = 0.25 * dt * (sigma**2 * j**2 + r * j)
        A = diags([-alpha[1:], 1 - beta, -gamma[:-1]], offsets=[-1, 0, 1], format="csc")
        B = diags([alpha[1:], 1 + beta, gamma[:-1]], offsets=[-1, 0, 1], format="csc")
        for step in range(N):
            tau_old = step * dt
            tau_new = (step + 1) * dt
            low_old = K * math.exp(-r * tau_old) if option_type == "put" else 0.0
            high_old = 0.0 if option_type == "put" else S_max - K * math.exp(-r * tau_old)
            low_new = K * math.exp(-r * tau_new) if option_type == "put" else 0.0
            high_new = 0.0 if option_type == "put" else S_max - K * math.exp(-r * tau_new)
            rhs = B @ V[1:M]
            rhs[0] += alpha[0] * (low_old + low_new)
            rhs[-1] += gamma[-1] * (high_old + high_new)
            V[1:M] = spsolve(A, rhs)
            V[0] = low_new; V[M] = high_new
    return {"S_grid": S_grid, "price_grid": V, "parameters": {"K": K, "T": T, "r": r, "sigma": sigma, "option_type": option_type, "S_max": S_max, "stock_steps": M, "time_steps": N, "method": method}}


def finite_difference_price_at(S0: float, **kwargs: Any) -> float:
    result = finite_difference_black_scholes_solver(**kwargs)
    return float(np.interp(S0, result["S_grid"], result["price_grid"]))


## Quantum Hamiltonian Simulation Utilities


In [ ]:
# Quantum grid, QFT, Hamiltonian, unitary dilation, post-selection, reconstruction, and resource utilities.

def make_log_price_grid(n_qubits: int, center_price: float, x_width: float = 0.80) -> Tuple[np.ndarray, np.ndarray, float]:
    N = 2 ** int(n_qubits)
    center = math.log(float(center_price))
    x_grid = np.linspace(center - x_width, center + x_width, N)
    S_grid = np.exp(x_grid)
    dx = float(x_grid[1] - x_grid[0]) if N > 1 else 1.0
    return x_grid, S_grid, dx


def make_symmetric_x_grid(n_qubits: int, x_max: float = 2.0) -> Tuple[np.ndarray, np.ndarray, float]:
    N = 2 ** int(n_qubits)
    x_grid = np.linspace(-x_max, x_max, N)
    S_grid = np.exp(x_grid)
    dx = float(x_grid[1] - x_grid[0]) if N > 1 else 1.0
    return x_grid, S_grid, dx


def payoff_vector(S_grid: np.ndarray, K: float, option_type: str = "call") -> np.ndarray:
    if normalize_option_type(option_type) == "call":
        return np.maximum(S_grid - K, 0.0).astype(float)
    return np.maximum(K - S_grid, 0.0).astype(float)


def normalize_state(vector: np.ndarray) -> Tuple[np.ndarray, float]:
    vector = np.asarray(vector, dtype=complex)
    norm_value = float(np.linalg.norm(vector))
    if norm_value <= 0:
        raise ValueError("Cannot normalize a zero payoff/state vector.")
    return vector / norm_value, norm_value


def duplicate_payoff_encoding(payoff: np.ndarray) -> np.ndarray:
    payoff = np.asarray(payoff, dtype=float)
    return np.concatenate([payoff, payoff[::-1]])


def qft_matrix(num_points: int) -> np.ndarray:
    N = int(num_points)
    j, k = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
    return np.exp(2j * np.pi * j * k / N) / math.sqrt(N)


def apply_qft_state(state: np.ndarray) -> np.ndarray:
    state = np.asarray(state, dtype=complex)
    return np.fft.ifft(state) * math.sqrt(len(state))


def apply_inverse_qft_state(state: np.ndarray) -> np.ndarray:
    state = np.asarray(state, dtype=complex)
    return np.fft.fft(state) / math.sqrt(len(state))


def momentum_eigenvalues(num_points: int, dx: float) -> np.ndarray:
    return -2.0 * np.pi * np.fft.fftfreq(int(num_points), d=float(dx))


def black_scholes_hamiltonian_diagonal(p_values: np.ndarray, sigma: float, r: float) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    return 1j * 0.5 * sigma**2 * p**2 - (0.5 * sigma**2 - r) * p + 1j * r


def hamiltonian_parts_diagonal(p_values: np.ndarray, sigma: float, r: float) -> Tuple[np.ndarray, np.ndarray]:
    p = np.asarray(p_values, dtype=float)
    hermitian = -(0.5 * sigma**2 - r) * p
    anti_hermitian = 1j * (0.5 * sigma**2 * p**2 + r)
    return hermitian, anti_hermitian


def commutator_norm_for_diagonals(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(np.diag(a) @ np.diag(b) - np.diag(b) @ np.diag(a)))


def non_unitary_anti_hermitian_diagonal(p_values: np.ndarray, tau: float, sigma: float, r: float) -> Tuple[np.ndarray, float]:
    contraction = np.exp(-float(tau) * (0.5 * sigma**2 * np.asarray(p_values, dtype=float)**2 + r))
    max_abs = float(np.max(np.abs(contraction)))
    scale = max(1.0, max_abs)
    return contraction / scale, scale


def dilation_matrix_from_diagonal(O_diag: np.ndarray) -> np.ndarray:
    O = np.asarray(O_diag, dtype=complex)
    if np.max(np.abs(O)) > 1 + 1e-10:
        raise ValueError("Dilation requires singular values <= 1. Scale O before constructing the block unitary.")
    complement = np.sqrt(np.maximum(0.0, 1.0 - np.abs(O)**2)).astype(complex)
    Omat = np.diag(O)
    Cmat = np.diag(complement)
    return np.block([[Omat, Cmat], [Cmat, -Omat.conjugate()]])


def dilation_unitarity_error(U: np.ndarray) -> float:
    identity = np.eye(U.shape[0], dtype=complex)
    return float(np.linalg.norm(U.conjugate().T @ U - identity))


def apply_dilation_and_postselect(O_diag: np.ndarray, state: np.ndarray) -> Dict[str, Any]:
    state = np.asarray(state, dtype=complex)
    norm_state = np.linalg.norm(state)
    if abs(norm_state - 1.0) > 1e-8:
        state = state / norm_state
    branch = np.asarray(O_diag, dtype=complex) * state
    probability = float(np.vdot(branch, branch).real)
    if probability <= 0:
        raise ValueError("Post-selection probability is zero.")
    post_selected = branch / math.sqrt(probability)
    return {"probability": probability, "post_selected_state": post_selected, "unnormalized_branch": branch}


def quantum_price_reconstruction(
    spot: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: str = "call",
    n_qubits: int = 8,
    x_width: float = 0.80,
    use_duplication: bool = False,
) -> Dict[str, Any]:
    x_grid, S_grid, dx = make_log_price_grid(n_qubits, center_price=spot, x_width=x_width)
    payoff = payoff_vector(S_grid, K, option_type)
    encoded = duplicate_payoff_encoding(payoff) if use_duplication else payoff
    state, payoff_norm = normalize_state(encoded)
    N = len(encoded)
    psi_p = apply_qft_state(state)
    p = momentum_eigenvalues(N, dx)
    hermitian, anti_hermitian = hamiltonian_parts_diagonal(p, sigma, r)
    phase = np.exp(1j * T * hermitian)
    O_scaled, scale = non_unitary_anti_hermitian_diagonal(p, T, sigma, r)
    evolved_pre = phase * psi_p
    post = apply_dilation_and_postselect(O_scaled, evolved_pre)
    recovered_p = post["post_selected_state"] * math.sqrt(post["probability"]) * scale
    recovered_state = apply_inverse_qft_state(recovered_p)
    recovered_curve_full = np.real(recovered_state) * payoff_norm
    if use_duplication:
        recovered_curve = recovered_curve_full[: len(S_grid)]
    else:
        recovered_curve = recovered_curve_full
    recovered_curve = np.where(np.abs(recovered_curve) < 1e-10, 0.0, recovered_curve)
    classical_curve = black_scholes_price(S_grid, K, T, r, sigma, option_type)
    price_at_spot = float(np.interp(spot, S_grid, recovered_curve))
    classical_at_spot = float(black_scholes_price(spot, K, T, r, sigma, option_type))
    return {
        "x_grid": x_grid,
        "S_grid": S_grid,
        "dx": dx,
        "payoff": payoff,
        "payoff_norm": payoff_norm,
        "momentum": p,
        "hermitian_diag": hermitian,
        "anti_hermitian_diag": anti_hermitian,
        "O_scaled": O_scaled,
        "nonunitary_scale": scale,
        "post_selection_probability": post["probability"],
        "price_curve": recovered_curve,
        "classical_curve": classical_curve,
        "price_at_spot": price_at_spot,
        "classical_at_spot": classical_at_spot,
        "use_duplication": use_duplication,
        "n_qubits": n_qubits,
    }


def curve_delta_gamma(S_grid: np.ndarray, price_curve: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    delta = np.gradient(price_curve, S_grid, edge_order=2)
    gamma = np.gradient(delta, S_grid, edge_order=2)
    return delta, gamma


def quantum_greeks_reconstruction(
    spot: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: str = "call",
    n_qubits: int = 7,
    x_width: float = 0.80,
    sigma_epsilon: float = 0.005,
    time_epsilon: float = 1 / 365,
    rate_epsilon: float = 0.0005,
) -> Dict[str, float]:
    base = quantum_price_reconstruction(spot, K, T, r, sigma, option_type, n_qubits, x_width)
    delta_curve, gamma_curve = curve_delta_gamma(base["S_grid"], base["price_curve"])
    delta = float(np.interp(spot, base["S_grid"], delta_curve))
    gamma = float(np.interp(spot, base["S_grid"], gamma_curve))
    up_sigma = quantum_price_reconstruction(spot, K, T, r, sigma + sigma_epsilon, option_type, n_qubits, x_width)["price_at_spot"]
    dn_sigma = quantum_price_reconstruction(spot, K, T, r, max(1e-6, sigma - sigma_epsilon), option_type, n_qubits, x_width)["price_at_spot"]
    vega = (up_sigma - dn_sigma) / (2 * sigma_epsilon)
    up_T = quantum_price_reconstruction(spot, K, T + time_epsilon, r, sigma, option_type, n_qubits, x_width)["price_at_spot"]
    dn_T = quantum_price_reconstruction(spot, K, max(1e-6, T - time_epsilon), r, sigma, option_type, n_qubits, x_width)["price_at_spot"]
    theta = (dn_T - up_T) / (2 * time_epsilon)
    up_r = quantum_price_reconstruction(spot, K, T, r + rate_epsilon, sigma, option_type, n_qubits, x_width)["price_at_spot"]
    dn_r = quantum_price_reconstruction(spot, K, T, r - rate_epsilon, sigma, option_type, n_qubits, x_width)["price_at_spot"]
    rho = (up_r - dn_r) / (2 * rate_epsilon)
    return {"Delta": delta, "Gamma": gamma, "Vega": float(vega), "Theta": float(theta), "Rho": float(rho), "post_selection_probability": base["post_selection_probability"], "price": base["price_at_spot"], "classical_price": base["classical_at_spot"]}


def quantum_price_for_position(position: Union[OptionPosition, Dict[str, Any]], n_qubits: int = 7, x_width: float = 0.80) -> float:
    pos = coerce_position(position)
    return quantum_price_reconstruction(pos.spot, pos.strike, pos.maturity, pos.risk_free_rate, pos.implied_volatility, pos.option_type, n_qubits, x_width)["price_at_spot"]


def value_portfolio_quantum(positions: Sequence[Union[OptionPosition, Dict[str, Any]]], n_qubits: int = 7, x_width: float = 0.80) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []
    for p in positions:
        pos = coerce_position(p)
        qprice = quantum_price_for_position(pos, n_qubits=n_qubits, x_width=x_width)
        qgreeks = quantum_greeks_reconstruction(pos.spot, pos.strike, pos.maturity, pos.risk_free_rate, pos.implied_volatility, pos.option_type, n_qubits=n_qubits, x_width=x_width)
        row = asdict(pos)
        row.update({"quantum_price": qprice, "position_quantum_value": qprice * pos.quantity})
        for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
            row[f"quantum_{greek}"] = qgreeks[greek]
            row[f"position_quantum_{greek}"] = qgreeks[greek] * pos.quantity
        row["post_selection_probability"] = qgreeks["post_selection_probability"]
        rows.append(row)
    df = pd.DataFrame(rows)
    totals = {"total_quantum_value": float(df["position_quantum_value"].sum())}
    for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
        totals[f"portfolio_quantum_{greek}"] = float(df[f"position_quantum_{greek}"].sum())
    return df, totals


def price_error_metrics(y_true: np.ndarray, y_pred: np.ndarray, S_grid: Optional[np.ndarray] = None, K: Optional[float] = None, option_type: str = "call") -> Dict[str, float]:
    true = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    err = pred - true
    metrics = {
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE": float(np.mean(np.abs(err))),
        "MaxAbsError": float(np.max(np.abs(err))),
        "RelativeErrorMean": float(np.mean(np.abs(err) / np.maximum(1.0, np.abs(true)))),
    }
    if S_grid is not None and K is not None:
        S = np.asarray(S_grid, dtype=float)
        atm_mask = np.abs(S / K - 1.0) <= 0.03
        if normalize_option_type(option_type) == "call":
            itm_mask = S > K * 1.03; otm_mask = S < K * 0.97
        else:
            itm_mask = S < K * 0.97; otm_mask = S > K * 1.03
        for label, mask in [("ATM", atm_mask), ("ITM", itm_mask), ("OTM", otm_mask)]:
            metrics[f"{label}ErrorMAE"] = float(np.mean(np.abs(err[mask]))) if np.any(mask) else np.nan
    return metrics


def statevector_memory_bytes(n_qubits: int, precision: str = "complex128") -> int:
    bytes_per_amp = 16 if precision == "complex128" else 8
    return int((2 ** int(n_qubits)) * bytes_per_amp)


def human_bytes(num_bytes: float) -> str:
    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    value = float(num_bytes)
    for unit in units:
        if value < 1000 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1000.0
    return f"{value:.2f} PB"


def resource_estimate_table(qubits: Sequence[int]) -> pd.DataFrame:
    rows = []
    for n in qubits:
        N = 2 ** int(n)
        mem128 = statevector_memory_bytes(n, "complex128")
        mem64 = statevector_memory_bytes(n, "complex64")
        qft_two_qubit = int(n * (n - 1) / 2)
        qft_single = int(n)
        rows.append({
            "grid_qubits": int(n),
            "grid_points": N,
            "complex128_bytes": mem128,
            "complex128_human": human_bytes(mem128),
            "complex64_bytes": mem64,
            "complex64_human": human_bytes(mem64),
            "hamiltonian_diagonal_entries": N,
            "qft_single_qubit_gate_estimate": qft_single,
            "qft_two_qubit_gate_estimate": qft_two_qubit,
            "small_circuit_depth_estimate": int(2 * n + qft_two_qubit),
            "runtime_proxy_N_log2N": float(N * n),
            "price_grid_resolution_proxy": float(1 / max(N - 1, 1)),
            "full_statevector_safe_default": bool(mem128 < 1_000_000_000),
        })
    return pd.DataFrame(rows)


def plot_heatmap(matrix: pd.DataFrame, title: str, filename: str) -> Path:
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(matrix.values.astype(float), cmap="RdYlGn", aspect="auto")
    ax.set_xticks(np.arange(matrix.shape[1])); ax.set_xticklabels(matrix.columns)
    ax.set_yticks(np.arange(matrix.shape[0])); ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Volatility shock")
    ax.set_ylabel("Spot shock")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="P&L")
    return save_current_figure(filename)


def plot_small_circuit_schematic(filename: str = "small_qft_circuit_schematic.png") -> Path:
    fig, ax = plt.subplots(figsize=(8, 3))
    qubits = ["q0", "q1", "q2", "ancilla"]
    for i, q in enumerate(qubits):
        y = len(qubits) - i
        ax.hlines(y, 0, 10, color="black", linewidth=1)
        ax.text(-0.4, y, q, va="center", ha="right")
    boxes = [(0.5, "Payoff\nprep"), (2.3, "QFT"), (4.1, "Phase"), (5.9, "Dilation"), (7.8, "IQFT"), (9.2, "Measure")]
    for x, label in boxes:
        ax.add_patch(plt.Rectangle((x, 1.25), 1.1, 3.1, fill=False, linewidth=1.5))
        ax.text(x + 0.55, 2.8, label, ha="center", va="center", fontsize=9)
    ax.set_xlim(-0.8, 10.5); ax.set_ylim(0.7, 4.7); ax.axis("off")
    ax.set_title("Small-n QFT/dilation prototype schematic")
    return save_current_figure(filename)


## Shared Validation Cell


In [ ]:
# Notebook validation helpers.

def validation_success(message: str) -> None:
    print(f"VALIDATION PASSED: {message}")


def run_shared_validations() -> None:
    S = 100.0; K = 100.0; T = 1.0; r = 0.05; sigma = 0.20
    assert abs(put_call_parity_error(S, K, T, r, sigma)) < 1e-8
    validation_success("put-call parity")
    call = black_scholes_price(S, K, T, r, sigma, "call")
    put = black_scholes_price(S, K, T, r, sigma, "put")
    assert call > 0 and put > 0 and call > put * 0.5
    validation_success("Black-Scholes price sanity")
    analytical = black_scholes_greeks(S, K, T, r, sigma, "call")
    fd = finite_difference_greeks(S, K, T, r, sigma, "call")
    assert abs(analytical["Delta"] - fd["Delta"]) < 1e-4
    assert abs(analytical["Gamma"] - fd["Gamma"]) < 1e-5
    validation_success("analytical Greeks versus finite-difference Greeks")
    x, Sg, dx = make_log_price_grid(3, center_price=100, x_width=0.5)
    payoff = payoff_vector(Sg, 100, "put")
    state, norm_value = normalize_state(payoff)
    assert abs(np.linalg.norm(state) - 1.0) < 1e-12 and norm_value > 0
    validation_success("payoff vector normalization")
    F = qft_matrix(8)
    assert np.linalg.norm(F.conjugate().T @ F - np.eye(8)) < 1e-10
    recovered = apply_inverse_qft_state(apply_qft_state(state))
    assert np.linalg.norm(recovered - state) < 1e-10
    validation_success("QFT unitarity and inverse QFT correctness")
    p = momentum_eigenvalues(8, dx)
    h = black_scholes_hamiltonian_diagonal(p, sigma, r)
    herm, anti = hamiltonian_parts_diagonal(p, sigma, r)
    assert h.shape == (8,) and herm.shape == anti.shape == (8,)
    assert commutator_norm_for_diagonals(herm, anti) < 1e-12
    validation_success("Hamiltonian dimensions and constant-volatility commutation")
    O = np.array([0.2, 0.5, 0.9, 1.0])
    U = dilation_matrix_from_diagonal(O)
    assert dilation_unitarity_error(U) < 1e-10
    post = apply_dilation_and_postselect(O, np.ones(4) / 2)
    assert 0 <= post["probability"] <= 1
    assert abs(np.linalg.norm(post["post_selected_state"]) - 1.0) < 1e-12
    validation_success("unitary dilation, post-selection probability, and recovered normalization")
    positions = load_portfolio_config()
    df, totals = value_portfolio(positions)
    assert abs(df["position_theoretical_value"].sum() - totals["total_theoretical_value"]) < 1e-8
    validation_success("portfolio value equals sum of position values")
    pnl = pd.Series([-10, -5, 0, 5, 15, -20, 30, -2, 4, 8], dtype=float)
    risk = var_expected_shortfall(pnl, levels=[0.95, 0.99])
    assert risk["ES_95"] >= risk["VaR_95"]
    assert risk["VaR_99"] >= risk["VaR_95"]
    validation_success("VaR and Expected Shortfall ordering")
    resources = resource_estimate_table([4, 40])
    mem40 = int(resources.loc[resources["grid_qubits"] == 40, "complex128_bytes"].iloc[0])
    assert mem40 > 10_000_000_000_000
    validation_success("resource estimator avoids impossible memory allocation")

run_shared_validations()
